In [1]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from ucimlrepo import fetch_ucirepo 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt

In [2]:
# fetch dataset 
car_evaluation = fetch_ucirepo(id=19) 
  
# data (as pandas dataframes) 
X = car_evaluation.data.features 
y = car_evaluation.data.targets 
  
# metadata 
print(car_evaluation.metadata) 
  
# variable information 
print(car_evaluation.variables) 


{'uci_id': 19, 'name': 'Car Evaluation', 'repository_url': 'https://archive.ics.uci.edu/dataset/19/car+evaluation', 'data_url': 'https://archive.ics.uci.edu/static/public/19/data.csv', 'abstract': 'Derived from simple hierarchical decision model, this database may be useful for testing constructive induction and structure discovery methods.', 'area': 'Other', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 1728, 'num_features': 6, 'feature_types': ['Categorical'], 'demographics': [], 'target_col': ['class'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 1988, 'last_updated': 'Thu Aug 10 2023', 'dataset_doi': '10.24432/C5JP48', 'creators': ['Marko Bohanec'], 'intro_paper': {'ID': 249, 'type': 'NATIVE', 'title': 'Knowledge acquisition and explanation for multi-attribute decision making', 'authors': 'M. Bohanec, V. Rajkovič', 'venue': '8th Intl Workshop on Expert Systems and their Applications, 

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from ucimlrepo import fetch_ucirepo

# 1. Logistic Regression
logistic_regression = LogisticRegression(
    max_iter=2000,
    random_state=1
)

# 2. Support Vector Machine
svm = SVC(
    C=1.0, 
    kernel='rbf', 
    degree=3, 
    gamma='auto',
    probability=True
)

# 3. Decision Tree
decision_tree = DecisionTreeClassifier(random_state=1)

# 4. Multi-Layer Perceptron
mlp = MLPClassifier(
    hidden_layer_sizes=(100,),
    max_iter=3000,
    random_state=1
)

In [9]:
# y is a one-column DataFrame; convert it to a Series
y = y.squeeze()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

models = {
    "Logistic Regression": logistic_regression,
    "SVM": svm,
    "Decision Tree": decision_tree,
    "MLP": mlp,
}

results = []

for name, model in models.items():
    # Fit the encoder using training data only
    pipeline = make_pipeline(
        OneHotEncoder(handle_unknown="ignore"),
        model
    )
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Macro F1": f1_score(y_test, predictions, average="macro"),
        "Weighted F1": f1_score(y_test, predictions, average="weighted"),
    })

    print(f"\n{name}")
    print(classification_report(y_test, predictions, zero_division=0))

results_df = pd.DataFrame(results).sort_values("Macro F1", ascending=False)
results_df


Logistic Regression
              precision    recall  f1-score   support

         acc       0.79      0.84      0.82       115
        good       0.69      0.43      0.53        21
       unacc       0.96      0.97      0.97       363
       vgood       0.95      0.90      0.92        20

    accuracy                           0.92       519
   macro avg       0.85      0.78      0.81       519
weighted avg       0.91      0.92      0.91       519


SVM
              precision    recall  f1-score   support

         acc       0.90      0.98      0.94       115
        good       1.00      0.90      0.95        21
       unacc       1.00      0.98      0.99       363
       vgood       0.90      0.90      0.90        20

    accuracy                           0.97       519
   macro avg       0.95      0.94      0.94       519
weighted avg       0.97      0.97      0.97       519


Decision Tree
              precision    recall  f1-score   support

         acc       0.92      0.89 

,Model,Accuracy,Macro F1,Weighted F1
3,MLP,0.990366,0.982084,0.990386
1,SVM,0.971098,0.944435,0.971457
2,Decision Tree,0.953757,0.922251,0.953638
0,Logistic Regression,0.915222,0.808307,0.912979


In [10]:
# y is a one-column DataFrame; convert it to a 1D Series
y_1d = y.squeeze()

# Reuse the same stratified five folds for every classifier
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
models = {
    "Logistic Regression": logistic_regression,
    "SVM": svm,
    "Decision Tree": decision_tree,
    "MLP": mlp,
}

cv_results = []
for name, model in models.items():
    pipeline = make_pipeline(
        OneHotEncoder(handle_unknown="ignore"),
        model
    )
    fold_accuracies = cross_val_score(
        pipeline, X, y_1d, cv=cv, scoring="accuracy", n_jobs=-1
    )
    cv_results.append({
        "Model": name,
        "Fold accuracies": fold_accuracies,
        "Mean accuracy": fold_accuracies.mean(),
        "Accuracy std": fold_accuracies.std(),
    })

results_df = pd.DataFrame(cv_results).sort_values(
    "Mean accuracy", ascending=False
)
results_df

,Model,Fold accuracies,Mean accuracy,Accuracy std
3,MLP,"[0.99711, 0.991329, 0.99422, 0.997101, 1.0]",0.995952,0.002947
1,SVM,"[0.953757, 0.979769, 0.976879, 0.962319, 0.982...",0.971066,0.011125
2,Decision Tree,"[0.968208, 0.971098, 0.973988, 0.950725, 0.971...",0.967007,0.008344
0,Logistic Regression,"[0.898844, 0.921965, 0.924855, 0.913043, 0.901...",0.912031,0.010489


## Compare feature importance across all four models

Permutation importance measures how much test accuracy drops when one original input feature is shuffled. Larger drops indicate greater reliance on that feature. The encoder stays inside each pipeline, so the six categorical columns are assessed as whole features.

In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.base import clone
import matplotlib.pyplot as plt

models_for_importance = {
    "Logistic Regression": logistic_regression,
    "SVM": svm,
    "Decision Tree": decision_tree,
    "MLP": mlp,
}

all_importances = []
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

for ax, (name, model) in zip(axes.flat, models_for_importance.items()):
    pipeline = make_pipeline(
        OneHotEncoder(handle_unknown="ignore"),
        clone(model)
    )
    pipeline.fit(X_train, y_train)
    importance = permutation_importance(
        pipeline, X_test, y_test,
        scoring="accuracy", n_repeats=30, random_state=42, n_jobs=-1
    )

    model_importance = pd.DataFrame({
        "Model": name,
        "Feature": X.columns,
        "Mean accuracy decrease": importance.importances_mean,
        "Std. dev.": importance.importances_std,
    }).sort_values("Mean accuracy decrease", ascending=False)
    all_importances.append(model_importance)

    plot_data = model_importance.sort_values("Mean accuracy decrease")
    ax.barh(
        plot_data["Feature"], plot_data["Mean accuracy decrease"],
        xerr=plot_data["Std. dev."]
    )
    ax.set_title(name)
    ax.set_xlabel("Accuracy decrease after shuffling")

plt.tight_layout()
plt.show()

importance_df = pd.concat(all_importances, ignore_index=True)
importance_df